<a href="https://colab.research.google.com/github/TsholoMolefe/Generative-AI/blob/main/ReactAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing Dependencies

In [1]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 10.9 MB/s eta 0:00:00


In [24]:
!pip install requests

In [2]:
import os, re, getpass
from groq import Groq

In [8]:
os.environ["GROQ_API_KEY"] = getpass.getpass("Provide your Groq API Key: ")

Provide your Groq API Key: ··········


In [9]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "groq/compound-mini"

In [10]:
models = client.models.list()

for model in models.data:
    print(model.id)

qwen/qwen3.6-27b
qwen/qwen3.8-27b
openai/gpt-oss-120b
allam-2-7b
whisper-large-v3-turbo
groq/compound-mini
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-v1-english
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-86m
groq/compound


Tools

In [39]:
def calculator(expression: str) -> str:
    """Evaluate a math expression and return the result as a string.

    Args:
        expression (str): Math expression, e.g. "23 * 47" or "(100 + 5) / 3".

    Returns:
        str: Numeric result, or an error message starting with "Error:".
    """
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


import requests

import requests

def get_weather(city: str) -> str:
    """Get real current weather for a city using Open-Meteo."""

    try:
        # Find the city's coordinates
        geo_url = "https://geocoding-api.open-meteo.com/v1/search"

        geo_params = {
            "name": city,
            "count": 1,
            "language": "en",
            "format": "json"
        }

        geo_response = requests.get(geo_url, params=geo_params)
        geo_response.raise_for_status()

        geo_data = geo_response.json()

        if "results" not in geo_data or not geo_data["results"]:
            return f"Could not find the city: {city}"

        location = geo_data["results"][0]

        latitude = location["latitude"]
        longitude = location["longitude"]
        city_name = location["name"]
        country = location.get("country", "")

        # Get current weather
        weather_url = "https://api.open-meteo.com/v1/forecast"

        weather_params = {
            "latitude": latitude,
            "longitude": longitude,
            "current": "temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,weather_code,wind_speed_10m",
            "temperature_unit": "celsius",
            "wind_speed_unit": "kmh",
            "timezone": "auto"
        }

        weather_response = requests.get(
            weather_url,
            params=weather_params
        )

        weather_response.raise_for_status()

        weather_data = weather_response.json()
        current = weather_data["current"]

        return (
            f"Weather in {city_name}, {country}: "
            f"{current['temperature_2m']}°C, "
            f"feels like {current['apparent_temperature']}°C, "
            f"humidity {current['relative_humidity_2m']}%, "
            f"wind {current['wind_speed_10m']} km/h, "
            f"precipitation {current['precipitation']} mm."
        )

    except Exception as e:
        return f"Error getting weather: {e}"

def word_count(text: str) -> str:
    """Count whitespace-separated words in a string.

    Args:
        text (str): Input text.

    Returns:
        str: Word count as a string.
    """
    return str(len(text.split()))

In [30]:
TOOLS = {
    "calculator": calculator,
    "get_weather": get_weather,
    "word_count": word_count
}

In [31]:
TOOL_DESCRIPTIONS = """
- calculator(expression: str) -> str
    Evaluate a math expression. Example: calculator("23 * 47")
- get_weather(city: str) -> str
    Return current weather for a city. Example: get_weather("Chennai")
- word_count(text: str) -> str
    Count words in text. Example: word_count("hello world")
"""

SYSTEM PROMPT

In [32]:
#Define thought/action/observation
SYSTEM_PROMPT = f"""You are a ReAct agent that solves problems step by step.

Tools available:
{TOOL_DESCRIPTIONS}

Format (follow exactly):

Thought: <reasoning>
Action: <tool_name>
Action Input: <input string>

After Action, STOP. The system replies with:

Observation: <result>

Continue with another Thought/Action, or finish:

Thought: I now know the final answer.
Final Answer: <answer>

Rules:
- One Thought + Action per turn, then wait.
- Action must be one of: {list(TOOLS.keys())}
- Action Input is a plain string (no quotes).
- Never invent Observations.
"""

In [33]:
print(SYSTEM_PROMPT)

You are a ReAct agent that solves problems step by step.

Tools available:

- calculator(expression: str) -> str
    Evaluate a math expression. Example: calculator("23 * 47")
- get_weather(city: str) -> str
    Return current weather for a city. Example: get_weather("Chennai")
- word_count(text: str) -> str
    Count words in text. Example: word_count("hello world")


Format (follow exactly):

Thought: <reasoning>
Action: <tool_name>
Action Input: <input string>

After Action, STOP. The system replies with:

Observation: <result>

Continue with another Thought/Action, or finish:

Thought: I now know the final answer.
Final Answer: <answer>

Rules:
- One Thought + Action per turn, then wait.
- Action must be one of: ['calculator', 'get_weather', 'word_count']
- Action Input is a plain string (no quotes).
- Never invent Observations.



In [34]:
#use this one
SYSTEM_PROMPT = f"""You are a ReAct agent that solves problems step by step.

Tools available:
{TOOL_DESCRIPTIONS}

You MUST follow this format exactly.

For a calculation or any task requiring a tool, your response MUST be:

Thought: <brief reasoning>
Action: <tool_name>
Action Input: <input string>

STOP after Action Input. Do NOT write Observation yourself.

The system will execute the tool and provide the result as:

Observation: <result>

After receiving an Observation, either use another tool or provide:

Thought: I now know the final answer.
Final Answer: <answer>

IMPORTANT RULES:
- Never write "Observation:" yourself.
- Never calculate a tool result yourself when a tool is available.
- Never give a Final Answer before receiving an Observation from the system.
- One Thought and one Action per turn.
- Action must be one of: {list(TOOLS.keys())}
- Action Input must be a plain string.
"""

Parser

In [35]:
def parse_response(text: str):
    """Parse LLM output into ('final', answer) | ('action', name, input) | ('error', msg)."""
    if m := re.search(r"Final Answer:\s*(.+)", text, re.DOTALL):
        return ("final", m.group(1).strip())

    a = re.search(r"Action:\s*(.+)", text)
    i = re.search(r"Action Input:\s*(.+)", text)
    if a and i:
        return ("action", a.group(1).strip(), i.group(1).strip().strip('"').strip("'"))

    return ("error", "Could not parse Action or Final Answer.")

AGENT LOOP

In [36]:
def run_agent(question: str, max_steps: int = 6, verbose: bool = True) -> str:
    """Run the ReAct loop until Final Answer or max_steps."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    if verbose:
        print(f"🧑 {question}\n" + "=" * 60)

    for step in range(1, max_steps + 1):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=0,
            stop=["Observation:"],
        )
        out = resp.choices[0].message.content.strip()
        messages.append({"role": "assistant", "content": out})
        if verbose:
            print(f"\n--- Step {step} ---\n🤖 {out}")

        parsed = parse_response(out)

        if parsed[0] == "final":
            if verbose: print(f"\n✅ {parsed[1]}")
            return parsed[1]
        if parsed[0] == "error":
            if verbose: print(f"⚠️ {parsed[1]}")
            return parsed[1]

        _, name, arg = parsed
        if name in TOOLS:
            try:    obs = TOOLS[name](arg)
            except Exception as e: obs = f"Error running {name}: {e}"
        else:
            obs = f"Error: unknown tool '{name}'. Available: {list(TOOLS.keys())}"

        if verbose: print(f"🔧 {obs}")
        messages.append({"role": "user", "content": f"Observation: {obs}"})

    return "Agent stopped: max_steps reached."

In [23]:
run_agent("What is 1 plus 3, then divided by 2?")

🧑 What is 1 plus 3, then divided by 2?

--- Step 1 ---
🤖 Thought: I need to compute (1+3)/2.
Action: calculator
Action Input: "(1+3)/2"
🔧 2.0

--- Step 2 ---
🤖 Thought: I now know the final answer.
Final Answer: 2.0

✅ 2.0


'2.0'

In [20]:
run_agent("What is 3 multiplied by 3, then divided by 3?")

🧑 What is 3 multiplied by 3, then divided by 3?

--- Step 1 ---
🤖 Thought: I now know the final answer.
Final Answer: 3

✅ 3


'3'

In [42]:
get_weather("Mumbai")

'Weather in Mumbai, India: 29.1°C, feels like 33.6°C, humidity 71%, wind 15.7 km/h, precipitation 0.2 mm.'

In [43]:
word_count("This is a test sentence.")

'5'